from https://github.com/PhilipsHue/PhilipsHueSDK-iOS-OSX/blob/00187a3db88dedd640f5ddfa8a474458dff4e1db/ApplicationDesignNotes/RGB%20to%20xy%20Color%20conversion.md


# Color to xy

We start with the color to xy conversion, which we will do in a couple of steps:

1. Get the RGB values from your color object and convert them to be between 0 and 1. So the RGB color (255, 0, 100) becomes (1.0, 0.0, 0.39)

2. Apply a gamma correction to the RGB values, which makes the color more vivid and more the like the color displayed on the screen of your device. This gamma correction is also applied to the screen of your computer or phone, thus we need this to create the same color on the light as on screen. This is done by the following formulas: float red = (red > 0.04045f) ? pow((red + 0.055f) / (1.0f + 0.055f), 2.4f) : (red / 12.92f); float green = (green > 0.04045f) ? pow((green + 0.055f) / (1.0f + 0.055f), 2.4f) : (green / 12.92f); float blue = (blue > 0.04045f) ? pow((blue + 0.055f) / (1.0f + 0.055f), 2.4f) : (blue / 12.92f);

3. Convert the RGB values to XYZ using the Wide RGB D65 conversion formula The formulas used:

    float X = red * 0.649926f + green * 0.103455f + blue * 0.197109f;

    float Y = red * 0.234327f + green * 0.743075f + blue * 0.022598f;

    float Z = red * 0.0000000f + green * 0.053077f + blue * 1.035763f;

4. Calculate the xy values from the XYZ values

    float x = X / (X + Y + Z);

    float y = Y / (X + Y + Z);

5. Check if the found xy value is within the color gamut of the light, if not continue with step 6, otherwise step 7 When we sent a value which the light is not capable of, the resulting color might not be optimal. Therefor we try to only sent values which are inside the color gamut of the selected light.

6. Calculate the closest point on the color gamut triangle and use that as xy value The closest value is calculated by making a perpendicular line to one of the lines the triangle consists of and when it is then still not inside the triangle, we choose the closest corner point of the triangle.

7. Use the Y value of XYZ as brightness The Y value indicates the brightness of the converted color.

# xy to color

The xy to color conversion is almost the same, but in reverse order.

1. Check if the xy value is within the color gamut of the lamp, if not continue with step 2, otherwise step 3 We do this to calculate the most accurate color the given light can actually do.

2. Calculate the closest point on the color gamut triangle and use that as xy value See step 6 of color to xy.

3. Calculate XYZ values Convert using the following formulas:

    float x = x; // the given x value

    float y = y; // the given y value

    float z = 1.0f - x - y;

    float Y = brightness; // The given brightness value

    float X = (Y / y) * x;

    float Z = (Y / y) * z;

4. Convert to RGB using Wide RGB D65 conversion (THIS IS A D50 conversion currently)

    float r = X * 1.4628067f - Y * 0.1840623f - Z * 0.2743606f;

    float g = -X * 0.5217933f + Y * 1.4472381f + Z * 0.0677227f;

    float b = X * 0.0349342f - Y * 0.0968930f + Z * 1.2884099f;

5. Apply reverse gamma correction

    r = r <= 0.0031308f ? 12.92f * r : (1.0f + 0.055f) * pow(r, (1.0f / 2.4f)) - 0.055f;

    g = g <= 0.0031308f ? 12.92f * g : (1.0f + 0.055f) * pow(g, (1.0f / 2.4f)) - 0.055f;

    b = b <= 0.0031308f ? 12.92f * b : (1.0f + 0.055f) * pow(b, (1.0f / 2.4f)) - 0.055f;

6. Convert the RGB values to your color object The rgb values from the above formulas are between 0.0 and 1.0.




In [10]:
def rgb_to_xy(r, g, b):
    
    # normalize
    red, green, blue = (v / 255 for v in [r, g, b])
    
    # gamma correction 
    red = pow((red + 0.055) / (1.0 + 0.055), 2.4) if (red > 0.04045) else (red / 12.92)
    green = pow((green + 0.055) / (1.0 + 0.055), 2.4) if (green > 0.04045) else (green / 12.92)
    blue = pow((blue + 0.055) / (1.0 + 0.055), 2.4) if (blue > 0.04045) else (blue / 12.92)
    
    
    # Wide RGB D65
    X = red * 0.649926 + green * 0.103455 + blue * 0.197109;
    Y = red * 0.234327 + green * 0.743075 + blue * 0.022598;
    Z = red * 0.0000000 + green * 0.053077 + blue * 1.035763;
    
    # calculate xy from XYZ
    x = X / (X + Y + Z);
    y = Y / (X + Y + Z);
    
    # TODO: check if the xy coordinate falls outside the bulb's color gamut. if so, find nearest matching color
    return (round(x*255), round(y*255))
    
    

In [11]:
rgb_to_xy(255, 0, 0)

(187, 68)

In [12]:
rgb_to_xy(0, 255, 0)

(29, 211)

In [13]:
rgb_to_xy(0, 0, 255)

(40, 5)